# W7-X with PolarFEEC

The Wendelstein 7-X equilibrium of DESC's examples, evaluated by [PolarFEEC.jl](https://gitlab.mpcdf.mpg.de/rwr/polarfeec.jl) through its Python wrapper `polarfeec`, and a neutral beam injected into it. PolarFEEC.jl reads GVEC and DESC equilibria in Julia and inverts their maps robustly, across the magnetic axis included. `PolarFEECEquilibrium` makes it a `MagneticEquilibrium`, so `deposit` and the flux shells work on it unchanged, as on the GVEC equilibrium of the [stellarator example](/freeofcharge/examples/stellarator-injection/).

`polarfeec` is installed from a checkout of PolarFEEC.jl: `pip install -e <PolarFEEC.jl>/python`. The first `import` installs Julia and compiles PolarFEEC.jl, which takes a while once; later runs reuse it.

## The equilibrium

DESC's W7-X example, a standard configuration with five field periods, loaded through `polarfeec.Equilibrium.from_desc` (PolarFEEC.jl reads DESC's own file, so the map is DESC's Zernike series, evaluated exactly).

In [ ]:
import time

import desc.examples
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots as pp
import polarfeec
import pyvista as pv

import freeofcharge as foc
from freeofcharge.desc_equilibrium import DESCEquilibrium
from freeofcharge.polarfeec_equilibrium import PolarFEECEquilibrium

desc_w7x = desc.examples.get("W7-X")
w7x_polarfeec = polarfeec.Equilibrium.from_desc(desc_w7x)
w7x = PolarFEECEquilibrium(w7x_polarfeec)
period = 2 * np.pi / w7x.nfp
print(isinstance(w7x, foc.MagneticEquilibrium), w7x.nfp, "field periods")

In 3D: the last closed surface over the whole torus, translucent and coloured by `|B|`, and four cross sections around it: bean-shaped where a field period starts (`φ = 0°, 216°`), triangular half a period later (`φ = 108°, 324°`). Click the picture for the interactive scene (vtk.js) and turn it.

In [ ]:
def surface_mesh(ds, rho=None, zeta=None):
    # A PyVista surface of ds at one rho (over theta, zeta) or one zeta (over rho, theta),
    # coloured by |B|, with the poloidal angle closed.
    selection = {"rho": rho} if rho is not None else {"zeta": zeta}
    part = ds.isel(selection)
    arrays = [part[c].values for c in ("X", "Y", "Z", "mod_B")]
    if rho is not None:  # (theta, zeta): close theta
        arrays = [np.concatenate([a, a[:1]], axis=0) for a in arrays]
    else:  # (rho, theta): close theta
        arrays = [np.concatenate([a, a[:, :1]], axis=1) for a in arrays]
    mesh = pv.StructuredGrid(*(a[..., None] for a in arrays[:3]))
    mesh["|B| [T]"] = arrays[3][..., None].ravel(order="F")
    return mesh


torus = w7x_polarfeec.evaluate(
    "mod_B", rho=[1.0], theta=64, zeta=np.linspace(0.0, 2 * np.pi, 64 * w7x.nfp + 1)
)
cuts = w7x_polarfeec.evaluate(
    "mod_B", rho=24, theta=96, zeta=[0.0, 1.5 * period, 3.0 * period, 4.5 * period]
)
clim = (float(cuts.mod_B.min()), float(cuts.mod_B.max()))
style = {"cmap": "viridis", "clim": clim, "scalars": "|B| [T]"}

plotter = pv.Plotter()
plotter.add_mesh(
    surface_mesh(torus, rho=0),
    opacity=0.3,
    smooth_shading=True,
    scalar_bar_args={"title": "field strength [T]"},
    **style,
)
for k in range(cuts.zeta.size):
    plotter.add_mesh(
        surface_mesh(cuts, zeta=k), lighting=False, show_scalar_bar=False, **style
    )
plotter.camera_position = [(10.0, -12.0, 10.0), (0.0, 0.0, -0.5), (0.0, 0.0, 1.0)]
pp.scene(
    plotter,
    alt="The W7-X last closed surface coloured by |B| with four cross sections",
    title="W7-X: last closed surface and cross sections",
)

The same three sections in the poloidal plane, with the flux-coordinate lines, and `|B|` on the surface `rho = 0.5` over one field period: the strong field in the bean-shaped section (`zeta = 0`) and the weak one in between make W7-X's mirror.

In [ ]:
sections = w7x_polarfeec.evaluate(
    "mod_B", rho=24, theta=97, zeta=[0.0, period / 4, period / 2]
)
fig, axes = plt.subplots(1, 3, figsize=(11, 4.2), layout="constrained", sharey=True)
for ax, zeta in zip(axes, sections.zeta.values, strict=True):
    sections.mod_B.plasma.plot.slice(
        coords="physical",
        plane="RZ",
        zeta=zeta,
        ax=ax,
        overlays={"coordinate_lines": {"rho": 5, "theta": 12}},
        title=f"φ = {np.degrees(zeta):.0f}°",
        xlabel="R [m]",
        ylabel="Z [m]" if ax is axes[0] else "",
    )
plt.show()

In [ ]:
surface = w7x_polarfeec.evaluate("mod_B", rho=[0.5], theta=96, zeta=96)
fig, ax = plt.subplots(figsize=(6.5, 3.8), layout="constrained")
surface.mod_B.plasma.plot.slice(
    x="zeta", y="theta", rho=0.5, ax=ax, title="|B| on rho = 0.5"
)
plt.show()

## The inverse map

A beam code works in Cartesian space, so every step needs `rho` at a point: the inverse of the map. `PolarFEECEquilibrium` agrees with DESC's own evaluation through freeofcharge's `DESCEquilibrium` adapter to roundoff, and is much faster (here per point through the wrapper; the batched evaluation of PolarFEEC.jl will be faster still).

In [ ]:
reference = DESCEquilibrium.from_example("W7-X")
lower, upper = w7x.bounds
points = lower + (upper - lower) * np.random.default_rng(1).random((2000, 3))
w7x.rho_at(points[:10]), reference.rho_at(points[:10])  # compile both once

start = time.perf_counter()
rho = np.asarray(w7x.rho_at(points))
polarfeec_time = time.perf_counter() - start
start = time.perf_counter()
rho_desc = np.asarray(reference.rho_at(points))
desc_time = time.perf_counter() - start

inside = rho < 1.0
field = np.asarray(w7x.field_at(points[inside]))
field_desc = np.asarray(reference.field_at(points[inside]))
print(
    f"{inside.sum()} of {len(points)} points inside, the same in both: {np.all(inside == (rho_desc < 1.0))}"
)
print(f"rho: largest difference {np.max(np.abs(rho[inside] - rho_desc[inside])):.1e}")
print(
    f"B:   largest relative difference {np.max(np.linalg.norm(field - field_desc, axis=1) / np.linalg.norm(field_desc, axis=1)):.1e}"
)
print(
    f"time: PolarFEEC {polarfeec_time:.2f} s, DESC adapter {desc_time:.2f} s ({desc_time / polarfeec_time:.0f}x)"
)

`rho` on a Cartesian grid in the bean-shaped plane, straight from `rho_at`: smooth across the magnetic axis, where flux coordinates are singular, and `inf` outside the last closed surface (white).

In [ ]:
R, Z = np.meshgrid(np.linspace(5.4, 6.3, 180), np.linspace(-0.95, 0.95, 190))
grid = np.stack([R.ravel(), np.zeros(R.size), Z.ravel()], axis=-1)
start = time.perf_counter()
rho_grid = np.asarray(w7x.rho_at(grid)).reshape(R.shape)
elapsed = time.perf_counter() - start
rho_grid[~np.isfinite(rho_grid)] = np.nan

edge = w7x_polarfeec.to_cartesian(
    np.stack([np.ones(257), np.linspace(0.0, 1.0, 257), np.zeros(257)], axis=-1)
)
axis = w7x_polarfeec.to_cartesian(np.array([0.0, 0.0, 0.0]))

fig, ax = plt.subplots(figsize=(5.2, 6), layout="constrained")
mesh = ax.pcolormesh(R, Z, rho_grid, cmap="viridis", shading="auto")
ax.contour(
    R, Z, rho_grid, levels=np.linspace(0.1, 0.9, 9), colors="white", linewidths=0.7
)
ax.plot(np.hypot(edge[:, 0], edge[:, 1]), edge[:, 2], color="black", linewidth=1.2)
ax.plot(
    np.hypot(*axis[:2]), axis[2], "+", color="red", markersize=12, label="magnetic axis"
)
fig.colorbar(mesh, ax=ax, label="rho")
ax.set(
    aspect="equal",
    xlabel="R [m]",
    ylabel="Z [m]",
    title=f"rho at {R.size} points ({elapsed:.1f} s)",
)
ax.legend(loc="lower left")
plt.show()

## Flux shells

`flux_surfaces` gives the shell volumes, cross sections, `|B|` with the averaging weights, the rotational transform and the trapped fraction, as for every `MagneticEquilibrium`. The total volume agrees with DESC's.

In [ ]:
rho_edges = np.linspace(0.0, 1.0, 11)
surfaces = w7x.flux_surfaces(rho_edges)
print(
    f"volume {surfaces.volume.sum():.4f} m^3, DESC {float(desc_w7x.compute('V')['V']):.4f} m^3"
)

fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
ax.plot(surfaces.rho, surfaces.rotational_transform, "o-", label="iota")
ax.plot(surfaces.rho, surfaces.trapped_fraction, "s-", label="trapped fraction")
ax.set(xlabel="rho", ylim=(0, 1.1))
ax.legend()
plt.show()

## Deposition

A hydrogen beam at 55 keV, the energy of W7-X's NBI, aimed tangentially at `R = 5.6 m` in the bean-shaped plane, into a parabolic plasma at two densities. Shine-through falls steeply with the density.

In [ ]:
origin, direction = foc.tangential_injection(5.6, 7.5, toroidal_angle=0.0)
beam = foc.NeutralBeam.gaussian(
    origin,
    direction,
    55e3,
    1.8e6,
    mass_number=1,
    width=0.1,
    divergence=0.01,
    markers=150,
    seed=1,
)
ions = {"H": (1, 1, 1.0)}
results = {}
for ne0 in (2e19, 6e19):
    plasma = foc.Plasma.parabolic(ne0, 3e3, ions=ions)
    results[ne0] = foc.deposit(
        beam, w7x, plasma, step=0.02, rho_edges=rho_edges, births_per_marker=1, seed=2
    )
    print(f"n_e0 = {ne0:.0e}: shine-through {results[ne0].shine_through_fraction:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.8), layout="constrained")
for ne0, color in zip(results, ("C0", "C1"), strict=True):
    ax.step(
        surfaces.rho,
        results[ne0].power_density(surfaces) / 1e3,
        where="mid",
        color=color,
        label=f"{ne0:.0e}",
    )
ax.set(xlabel="rho", ylabel="deposited power [kW/m^3]")
ax.legend(title="n_e0 [m^-3]")
plt.show()

The beam in the machine: every tenth beam line and the births at the lower density, with the last closed surface and the bean-shaped section.

In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(
    surface_mesh(torus, rho=0),
    opacity=0.25,
    smooth_shading=True,
    scalar_bar_args={"title": "field strength [T]"},
    **style,
)
plotter.add_mesh(
    surface_mesh(cuts, zeta=0), lighting=False, show_scalar_bar=False, **style
)
starts = np.asarray(beam.markers.positions)
directions = np.asarray(beam.directions)
for k in range(0, len(starts), 10):
    plotter.add_mesh(
        pv.Line(starts[k], starts[k] + 9.0 * directions[k]), color="black", line_width=1
    )
plotter.add_points(
    np.asarray(results[2e19].births.positions),
    color="orange",
    point_size=7,
    render_points_as_spheres=True,
)
plotter.camera_position = [(13.0, -6.0, 6.0), (4.5, 1.0, -0.3), (0.0, 0.0, 1.0)]
pp.scene(
    plotter,
    alt="W7-X's last closed surface, beam lines and the beam's births",
    title="Beam births in W7-X",
)